# Week 10: Scope & Mini-Library — PHASE 3: Build a checkable report

*📚 Computer Programming I · ⏱️ 5 Hours · 👨‍🏫 Dr. Arif Solmaz*

## Use the same function with two sensors

We have a reusable conversion. Reuse breaks down if the function quietly uses whichever sensor settings were changed last.

Sensor A has offset 0.5 V and sensor B has offset 1.0 V. Both use gain 25 N/V. Each reports 2.5 V.

**Try this first — before code.** Calculate each force on paper. Should processing B first change the result for A? Explain what information each calculation needs.

**Why this week's tool?** Parameters pass the settings openly. Local variables keep each call’s work separate. So one small library can serve several sensors without hidden changes.

**By the end.** Process A, then B, then A again. Explain why A’s two results agree.


## Explain the program: Scope finds names; mutation changes objects

A function can give a local name a new value without changing the caller’s name. But a parameter can point to the caller’s list. Changing that shared list changes what the caller sees. “Local variable” does not mean “private copy of every object.”

**Draw or trace.** Draw a box for the caller and a box for the function. In each box, draw a name pointing to one list [2, 4]. Compare adding an element to that list with pointing the local name at a new list.

**Predict before running.** Which changes the caller’s list: values.append(6), or values = [6] inside the function?

<details><summary>Trace and explanation — after your prediction</summary>

1. append changes the shared list, so the caller sees [2, 4, 6].
2. The assignment values = [6] points the local name at a new list.
3. The caller still has its original list unless the function returns the new one and the caller stores it.

The first operation changes shared data. The second only changes which list the local name points to. When you build a reusable library, write down which functions change their inputs. Pass each sensor’s settings as arguments.

</details>

**Change one thing.** Use the same helper with two sensor datasets in different orders. Predict which hidden global value could make the result depend on call order.

**Türkçe:** Kapsam adın nerede bulunduğunu belirler; mutasyon nesneyi değiştirir. Yerel parametre çağıranın nesnesini paylaşabilir.


<details><summary>Learning objectives</summary>

## 🎯 Learning Objectives

By the end of this week, you will be able to:

- Understand the difference between **local** and **global** scope
- Explain the **lifetime** of variables inside functions
- Use the `global` keyword (and know when to avoid it)
- Recognize and resolve **variable shadowing** issues
- Build functions that **call other functions** (composition)
- Organize code into a **mini-library** of reusable functions
- **Refactor** a flat script into well-structured functions

</details>


<details><summary>Class participation and assessment</summary>

---
## 🤝 Engineering Learning Contract

- **Professional relevance:** examples and core exercises model the data, sensing, automation, numerical, and decision tasks used in engineering.
- **Interaction:** predict before running, compare reasoning with a partner, and ask whenever a step is unclear; scheduled checkpoints guarantee question time.
- **Assessment alignment:** worked examples and Core Exercises 1–8 rehearse the same reasoning operations used on exams—trace, implement, debug, interpret, and justify—while exam values and contexts may change.
- **Assessment:** Assessment consists only of the midterm exam (50%) and final exam (50%). Weekly notebooks, exercises, projects, demonstrations and presentations are ungraded practice; no weekly submission is required.

</details>


<details><summary>Class schedule and checkpoints</summary>

---
## 🧭 Five-Hour Class Roadmap

This notebook is designed for one five-hour class with four short breaks.

| Target | Activity |
|---|---|
| 00:00–00:55 | Concepts and examples → Checkpoint 1 |
| 00:55–01:05 | Break |
| 01:05–01:55 | Concepts and examples → Checkpoint 2 |
| 01:55–02:05 | Break |
| 02:05–02:55 | Concepts and examples → Checkpoint 3 |
| 02:55–03:05 | Break |
| 03:05–03:55 | Concepts and examples → Checkpoint 4 |
| 03:55–04:05 | Break |
| 04:05–04:45 | Core Practice (Exercises 1–8) → Checkpoint 5 |
| 04:45–05:00 | Review and retry failed checks |

Concept checkpoints compare your predictions with an expected answer. Checkpoint 5 is your practice reflection. No grading submission is sent by these tools. Save the notebook to keep your work. Exercises 9 and above are optional extensions.

</details>


In [1]:
_checkpoint_results = {}
def check_answer(number, answer, expected, explanation):
    actual = str(answer).strip().lower().replace(" ", "")
    target = str(expected).strip().lower().replace(" ", "")
    correct = actual == target
    _checkpoint_results[int(number)] = ("Concept check", int(correct), 1)
    if correct:
        print(f"Checkpoint {number}: correct. {explanation}")
    elif not str(answer).strip():
        print(f"Checkpoint {number}: enter your prediction, then run again.")
    else:
        print(f"Checkpoint {number}: review the example and try again.")
    return correct
def record_checkpoint(number, checks):
    """Report each concrete concept check used by the introductory notebook."""
    passed = sum(bool(correct) for _, correct in checks)
    _checkpoint_results[int(number)] = ("Concept checks", passed, len(checks))
    print(f"Checkpoint {number}: {passed}/{len(checks)} concept checks match.")
    for label, correct in checks:
        print(("OK: " if correct else "Review: ") + label)
    return passed, len(checks)
def exercise_checkpoint(number, practiced, expected=8):
    """Summarize an explicit self-report; this does not grade your code."""
    if not isinstance(practiced, (list, tuple, set)):
        _checkpoint_results.pop(int(number), None)
        print("Use a list of exercise numbers, for example [1, 2].")
        return 0, expected
    if any(type(item) is not int or not 1 <= item <= expected for item in practiced):
        _checkpoint_results.pop(int(number), None)
        print(f"Use whole exercise numbers from 1 to {expected}.")
        return 0, expected
    done = set(practiced)
    _checkpoint_results[int(number)] = ("Practice self-report", len(done), expected)
    print(f"Practice self-report: {len(done)}/{expected} core exercises reviewed.")
    print("This is your reflection, not a correctness score or a grade.")
    remaining = [str(i) for i in range(1, expected + 1) if i not in done]
    if remaining:
        print("Still to review:", ", ".join(remaining))
    print("For each exercise: test the result, explain the steps, then compare with the worked solution.")
    return len(done), expected
def show_progress_summary():
    print("\nMy study feedback (this runtime)")
    for number in range(1, 6):
        if number in _checkpoint_results:
            kind, count, total = _checkpoint_results[number]
            print(f"{number}. {kind}: {count}/{total}")
        else:
            print(f"{number}. Not run yet")
    print("These checks send no grading submission. Save your notebook to keep your work.")
print("Local study tools ready.")


Local study tools ready.


**Line-by-line walkthrough**

- **Line 1 —** `` _checkpoint_results = {} ``

  Create an empty dictionary to remember checkpoint results during this run. Nothing has been checked yet.

- **Line 2 —** `` def check_answer(number, answer, expected, explanation): ``

  Define a reusable function with four inputs: checkpoint number, your answer, the expected answer and an explanation. Its indented body runs only when called.

- **Line 3 —** `` actual = str(answer).strip().lower().replace(" ", "") ``

  Convert your answer to text, trim whitespace at its ends, lowercase it and remove ordinary spaces. Thus `2, 4, 6` can match `2,4,6`.

- **Line 4 —** `` target = str(expected).strip().lower().replace(" ", "") ``

  Apply the same text cleanup to the expected answer, so the comparison uses the same rules on both sides.

- **Line 5 —** `` correct = actual == target ``

  Compare the cleaned strings with `==`; store either `True` or `False` in `correct`. This compares text, not mathematical equivalence.

- **Line 6 —** `` _checkpoint_results[int(number)] = ("Concept check", int(correct), 1) ``

  Use the integer checkpoint number as a dictionary key. Store a label, a score of 1 or 0, and a possible total of 1; rerunning replaces this checkpoint's earlier result.

- **Line 7 —** `` if correct: ``

  Choose the following indented branch only if the comparison was true.

- **Line 8 —** `` print(f"Checkpoint {number}: correct. {explanation}") ``

  Print the checkpoint number and its explanation. The braces insert the values supplied to the function.

- **Line 9 —** `` elif not str(answer).strip(): ``

  If the answer was not correct, check whether it was blank after trimming whitespace; `not` reverses the empty string's false truth value.

- **Line 10 —** `` print(f"Checkpoint {number}: enter your prediction, then run again.") ``

  For a blank answer, ask the learner to enter a prediction. This branch does not claim the answer is wrong.

- **Line 11 —** `` else: ``

  Choose this final branch only when the answer was nonblank and did not match.

- **Line 12 —** `` print(f"Checkpoint {number}: review the example and try again.") ``

  Print a request to revisit the example; this is feedback, not an exam grade.

- **Line 13 —** `` return correct ``

  Return the Boolean result to the caller and finish this function call. Printing and returning serve different purposes.

- **Line 14 —** `` def record_checkpoint(number, checks): ``

  Define a second helper that receives a checkpoint number and pairs of labels and check results.

- **Line 15 —** `` """Report each concrete concept check used by the introductory notebook.""" ``

  This quoted text documents the function. It is not a command and produces no printed output.

- **Line 16 —** `` passed = sum(bool(correct) for _, correct in checks) ``

  For each `(label, correct)` pair, turn its result into True/False, then add those Booleans. Each True contributes 1; each False contributes 0.

- **Line 17 —** `` _checkpoint_results[int(number)] = ("Concept checks", passed, len(checks)) ``

  Save the count passed and the number of checks under this checkpoint number. `len(checks)` counts the supplied pairs.

- **Line 18 —** `` print(f"Checkpoint {number}: {passed}/{len(checks)} concept checks match.") ``

  Print how many checks passed out of the number supplied, using the calculated counts.

- **Line 19 —** `` for label, correct in checks: ``

  Loop through each pair, unpacking its description into `label` and its result into `correct`.

- **Line 20 —** `` print(("OK: " if correct else "Review: ") + label) ``

  Choose `OK: ` for a true result or `Review: ` for a false one, join it to the label and print one line per check.

- **Line 21 —** `` return passed, len(checks) ``

  Return a pair: the number passed and the number supplied. The comma creates the tuple used by the caller.

- **Line 22 —** `` def exercise_checkpoint(number, practiced, expected=8): ``

  Define a reflection helper; if its third argument is omitted, it assumes eight core exercises.

- **Line 23 —** `` """Summarize an explicit self-report; this does not grade your code.""" ``

  Document that this helper records the learner's own report, rather than checking their exercise solutions.

- **Line 24 —** `` if not isinstance(practiced, (list, tuple, set)): ``

  Check that the supplied reflection is a list, tuple or set. A string or a single number does not meet this input contract.

- **Line 25 —** `` _checkpoint_results.pop(int(number), None) ``

  Remove any old entry for this checkpoint. The default `None` means removing a nonexistent key is harmless.

- **Line 26 —** `` print("Use a list of exercise numbers, for example [1, 2].") ``

  Tell the learner to provide a collection such as `[1, 2]`.

- **Line 27 —** `` return 0, expected ``

  Return `(0, expected)` immediately; this early return prevents invalid data reaching later lines.

- **Line 28 —** `` if any(type(item) is not int or not 1 <= item <= expected for item in practiced): ``

  Examine each item. Reject if any item is not an ordinary integer or lies outside 1 through `expected`; short-circuiting avoids comparing a noninteger to those bounds.

- **Line 29 —** `` _checkpoint_results.pop(int(number), None) ``

  Remove the earlier result for this checkpoint so invalid new input cannot leave an old completion claim in place.

- **Line 30 —** `` print(f"Use whole exercise numbers from 1 to {expected}.") ``

  Print the permitted whole-number range, inserting the value of `expected`.

- **Line 31 —** `` return 0, expected ``

  Return immediately with zero reported completions and the expected number of exercises.

- **Line 32 —** `` done = set(practiced) ``

  Convert the collection to a set to remove duplicates. Reporting exercise 1 twice still counts as one exercise.

- **Line 33 —** `` _checkpoint_results[int(number)] = ("Practice self-report", len(done), expected) ``

  Store the number of distinct exercises the learner reported, together with the expected total and a reflection label.

- **Line 34 —** `` print(f"Practice self-report: {len(done)}/{expected} core exercises reviewed.") ``

  Print that distinct count as a self-report, rather than treating the number of cell executions as practice completed.

- **Line 35 —** `` print("This is your reflection, not a correctness score or a grade.") ``

  Explicitly tell the learner that this is neither an automatic correctness check nor a grade.

- **Line 36 —** `` remaining = [str(i) for i in range(1, expected + 1) if i not in done] ``

  Build a list of exercise numbers not in `done`, converting each missing number to text so it can later be joined into a message.

- **Line 37 —** `` if remaining: ``

  Check whether that list contains any items. An empty list is false, so no reminder is needed when all exercises were reported.

- **Line 38 —** `` print("Still to review:", ", ".join(remaining)) ``

  Join the missing numbers with comma-space separators and print them in a single reminder line.

- **Line 39 —** `` print("For each exercise: test the result, explain the steps, then compare with the worked solution.") ``

  Print the same study reminder regardless of how many exercises were reported; this line is outside the preceding `if`.

- **Line 40 —** `` return len(done), expected ``

  Return the distinct number reported and the expected total as a pair.

- **Line 41 —** `` def show_progress_summary(): ``

  Define a helper with no arguments that displays the currently stored checkpoint results.

- **Line 42 —** `` print("\nMy study feedback (this runtime)") ``

  Print a blank line followed by the summary heading; `\n` starts a new output line.

- **Line 43 —** `` for number in range(1, 6): ``

  Visit checkpoint numbers 1 through 5. The stop value 6 is excluded.

- **Line 44 —** `` if number in _checkpoint_results: ``

  Check whether this checkpoint has an entry in the dictionary before trying to read it.

- **Line 45 —** `` kind, count, total = _checkpoint_results[number] ``

  Unpack the stored three-item tuple into its label, count and total.

- **Line 46 —** `` print(f"{number}. {kind}: {count}/{total}") ``

  Print the recorded label and fraction for this checkpoint.

- **Line 47 —** `` else: ``

  Choose this branch if the checkpoint number was absent from the dictionary.

- **Line 48 —** `` print(f"{number}. Not run yet") ``

  Print that this particular checkpoint has not been run yet.

- **Line 49 —** `` print("These checks send no grading submission. Save your notebook to keep your work.") ``

  After all five numbers, print one reminder that this local feedback sends no grading submission.

- **Line 50 —** `` print("Local study tools ready.") ``

  Run this print immediately when the setup cell executes. The function definitions above have been created, but their bodies have not yet been called.



This week uses dictionaries and tuple returns from Week 7.


---
## Part 1: Review — Functions

Before we explore scope, let's quickly recap what we know about functions.

A **function** is a reusable block of code that:
- Is defined with `def`
- Can accept **parameters** (inputs)
- Can **return** a value (output)

| Concept | Syntax | Example |
|---------|--------|---------|
| Define a function | `def name(params):` | `def greet(name):` |
| Call a function | `name(args)` | `greet("Elif")` |
| Return a value | `return value` | `return total` |
| Default parameter | `def f(x=10):` | `def greet(name="Student"):` |

**Figure 1.1: A simple function with parameters and return value**

In [2]:
def calculate_average(a, b, c):
    """Calculate the average of three numbers."""
    total = a + b + c
    average = total / 3
    return average
result = calculate_average(85, 90, 78)
print(f"Average: {result:.1f}")


Average: 84.3


**Line-by-line walkthrough**

- **Line 1 —** `` def calculate_average(a, b, c): ``

  Define a function receiving three numerical arguments.

- **Line 2 —** `` """Calculate the average of three numbers.""" ``

  Document that this function calculates their arithmetic mean.

- **Line 3 —** `` total = a + b + c ``

  Add the parameters and store the sum in a local variable.

- **Line 4 —** `` average = total / 3 ``

  Divide by three because this function always receives three numbers.

- **Line 5 —** `` return average ``

  Return the locally calculated mean to the caller.

- **Line 6 —** `` result = calculate_average(85, 90, 78) ``

  Call with the three supplied values and store the result outside the function.

- **Line 7 —** `` print(f"Average: {result:.1f}") ``

  Print the result to one decimal place; this changes its display, not its stored precision.



**Figure 1.2: Functions with default parameters**

In [3]:
def greet(name, greeting="Hello"):
    """Greet a person with a custom or default greeting."""
    return f"{greeting}, {name}!"
print(greet("Ayse"))
print(greet("Mehmet", "Welcome"))


Hello, Ayse!
Welcome, Mehmet!


**Line-by-line walkthrough**

- **Line 1 —** `` def greet(name, greeting="Hello"): ``

  Define a required name parameter and a greeting with a Hello default.

- **Line 2 —** `` """Greet a person with a custom or default greeting.""" ``

  Document the function's greeting behavior.

- **Line 3 —** `` return f"{greeting}, {name}!" ``

  Return the formatted greeting without printing inside the function.

- **Line 4 —** `` print(greet("Ayse")) ``

  Omit the optional argument to use Hello, then print the returned text.

- **Line 5 —** `` print(greet("Mehmet", "Welcome")) ``

  Supply Welcome to replace the default for this call and print the result.



> 💡 **Note:** Functions help us follow the **DRY principle** — Don't Repeat Yourself. If you write the same code more than once, it probably belongs in a function.

---
## Part 2: What is Scope?

**Scope** determines which binding a name refers to. A local name belongs to a function call; a global name belongs to the module (or notebook runtime). Different names can refer to the same object.

| Scope | Where bound | Lookup |
|---|---|---|
| Local | Parameters and ordinary assignments in a function | Within that function; enclosing bindings can also be used by nested functions |
| Global | At module/notebook level | Available to functions unless a nearer binding shadows it |

A global is not automatically shared across every Python module. A name must already be bound when it is read.


**Figure 2.1: Visualizing scope**

In [4]:
university = "ISTUN"
def show_info():
    department = "Mechatronics Engineering"
    print(f"{department} at {university}")
show_info()
print(university)


Mechatronics Engineering at ISTUN
ISTUN


**Line-by-line walkthrough**

- **Line 1 —** `` university = "ISTUN" ``

  Bind university in the notebook's surrounding namespace.

- **Line 2 —** `` def show_info(): ``

  Define a function with no parameters.

- **Line 3 —** `` department = "Mechatronics Engineering" ``

  Create department locally when the function is called.

- **Line 4 —** `` print(f"{department} at {university}") ``

  Print both names: department resolves locally and university resolves in the surrounding namespace.

- **Line 5 —** `` show_info() ``

  Call the function, creating its local department and producing the combined message.

- **Line 6 —** `` print(university) ``

  Print university outside the function. department is not created here; reading it in a fresh surrounding namespace would raise NameError.



> 💡 **Note:** Python looks for variables in this order: **Local → Enclosing → Global → Built-in** (the LEGB rule). For now, focus on Local and Global.

---
### ⏱️ Checkpoint 1 of 5 — Scope (target 00:55)

Is a variable created inside a function normally local or global?

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [5]:
checkpoint_1_answer = ""
check_answer(
    1, checkpoint_1_answer, 'local',
    'Ordinary function assignments create local bindings; global and nonlocal declarations are exceptions.',
)


Checkpoint 1: enter your prediction, then run again.


**Line-by-line walkthrough**

- **Line 1 —** `` checkpoint_1_answer = "" ``

  Store your response as text; replace the empty string before running the check.

- **Line 2 —** `` check_answer( ``

  Begin a call to the study helper defined earlier.

- **Line 3 —** `` 1, checkpoint_1_answer, 'local', ``

  Supply checkpoint 1, your response and expected answer text `local`.

- **Line 4 —** `` 'Ordinary function assignments create local bindings; global and nonlocal declarations are exceptions.', ``

  Supply the feedback explanation: Ordinary function assignments create local bindings unless declarations such as global or nonlocal specify otherwise.

- **Line 5 —** `` ) ``

  Close the call to print feedback and record the result.



---
## Part 3: Local Scope

Each ordinary function call has its own local bindings. Parameters are bound when the call starts; other local names receive values as assignments execute. The caller cannot access them just by using the same name.

When a call finishes, its ordinary local workspace is no longer available to the caller. **This does not mean every referenced object is destroyed:** a returned list or an object also referenced elsewhere can remain alive. Nested functions can also retain enclosing bindings (a later topic).


**Figure 3.1: Local variables have limited lifetime**

In [6]:
def calculate_bonus(salary):
    bonus_rate = 0.15
    bonus = salary * bonus_rate
    return bonus
result = calculate_bonus(5000)
print(f"Bonus: {result}")


Bonus: 750.0


**Line-by-line walkthrough**

- **Line 1 —** `` def calculate_bonus(salary): ``

  Define a bonus calculation receiving salary as an argument.

- **Line 2 —** `` bonus_rate = 0.15 ``

  Create a local rate of 0.15 for this call.

- **Line 3 —** `` bonus = salary * bonus_rate ``

  Calculate the local bonus from salary and rate.

- **Line 4 —** `` return bonus ``

  Return the numerical bonus. The local names are not thereby created in the caller's namespace.

- **Line 5 —** `` result = calculate_bonus(5000) ``

  Call with 5000 and store the returned 750.0 under the caller's result name.

- **Line 6 —** `` print(f"Bonus: {result}") ``

  Print that result. Reading bonus_rate or bonus outside the function would fail unless an independent binding already existed there.



**Figure 3.2: Each function call creates its own local variables**

In [7]:
def count_down(start):
    counter = start
    while counter > 0:
        print(counter, end=" ")
        counter -= 1
    print("Go!")
count_down(3)
count_down(5)


3 2 1 Go!
5 4 3 2 1 Go!


**Line-by-line walkthrough**

- **Line 1 —** `` def count_down(start): ``

  Define a countdown function with a starting-value parameter.

- **Line 2 —** `` counter = start ``

  Initialize a local counter anew on every call.

- **Line 3 —** `` while counter > 0: ``

  Continue while this call's counter is positive.

- **Line 4 —** `` print(counter, end=" ") ``

  Print its current value followed by a space.

- **Line 5 —** `` counter -= 1 ``

  Decrease the local counter so the loop progresses toward zero.

- **Line 6 —** `` print("Go!") ``

  Print Go! after the loop, ending the output line.

- **Line 7 —** `` count_down(3) ``

  Call with three, producing 3 2 1 Go!.

- **Line 8 —** `` count_down(5) ``

  Call with five. This call gets a fresh counter and is unaffected by the first call's final zero.



**Figure 3.3: Parameters are also local variables**

In [8]:
def double(x):
    x = x * 2
    return x
number = 10
result = double(number)
print(f"number = {number}")
print(f"result = {result}")


number = 10
result = 20


**Line-by-line walkthrough**

- **Line 1 —** `` def double(x): ``

  Define a function with a local parameter x.

- **Line 2 —** `` x = x * 2 ``

  Compute twice its value and rebind local x to that new number; the caller's binding is unaffected.

- **Line 3 —** `` return x ``

  Return the doubled value.

- **Line 4 —** `` number = 10 ``

  Bind number to 10 in the caller's namespace.

- **Line 5 —** `` result = double(number) ``

  Pass its value into double and store the returned 20 separately.

- **Line 6 —** `` print(f"number = {number}") ``

  Print number, which is still 10.

- **Line 7 —** `` print(f"result = {result}") ``

  Print result, which holds the doubled value.



### Names, rebinding, and mutation

A parameter is a **new local name for the object passed by the caller**. Python does
not automatically copy that object. `number = number + 1` rebinds the local name to a
new integer; `items.append(3)` mutates the same list the caller can see.
`items = [99]` then rebinds only the local name to a different list.

| Step | Caller `values` | Local `items` |
|---|---|---|
| Call function | `[1, 2]` | Same list `[1, 2]` |
| `items.append(3)` | `[1, 2, 3]` | Same list `[1, 2, 3]` |
| `items = [99]` | `[1, 2, 3]` | A different list `[99]` |

**Türkçe:** Yerel adı başka nesneye bağlamak ile ortak listedeki elemanları
değiştirmek farklı işlemlerdir. Listeyi değiştirmek çağıran kodu da etkiler.


In [9]:
def rebind_number(number):
    number = number + 1
    print("Inside number:", number)
def mutate_then_rebind(items):
    items.append(3)
    print("After append:", items)
    items = [99]
    print("After local rebinding:", items)
number = 10
rebind_number(number)
print("Caller number:", number)
values = [1, 2]
mutate_then_rebind(values)
print("Caller list:", values)


Inside number: 11
Caller number: 10
After append: [1, 2, 3]
After local rebinding: [99]
Caller list: [1, 2, 3]


**Line-by-line walkthrough**

- **Line 1 —** `` def rebind_number(number): ``

  Define a function receiving a number.

- **Line 2 —** `` number = number + 1 ``

  Rebind its local parameter to the incremented number; this does not change the caller's name.

- **Line 3 —** `` print("Inside number:", number) ``

  Print the local value, 11 for the demonstration.

- **Line 4 —** `` def mutate_then_rebind(items): ``

  Define a function receiving a list reference.

- **Line 5 —** `` items.append(3) ``

  Append 3 to that list object. The caller sees this mutation because both names initially refer to the same list.

- **Line 6 —** `` print("After append:", items) ``

  Print the shared list after mutation.

- **Line 7 —** `` items = [99] ``

  Rebind only the local name items to a new list containing 99; the caller still refers to the original mutated list.

- **Line 8 —** `` print("After local rebinding:", items) ``

  Print this new local list.

- **Line 9 —** `` number = 10 ``

  Set the caller's number to 10.

- **Line 10 —** `` rebind_number(number) ``

  Call the number function, which prints its local incremented value.

- **Line 11 —** `` print("Caller number:", number) ``

  Print the caller's unchanged number.

- **Line 12 —** `` values = [1, 2] ``

  Create the caller's list containing 1 and 2.

- **Line 13 —** `` mutate_then_rebind(values) ``

  Call the list function, which mutates the shared list before rebinding its local name.

- **Line 14 —** `` print("Caller list:", values) ``

  Print the caller's list, now 1, 2, 3; local rebinding did not replace it with 99.



---
## Part 4: Global Scope

Names assigned at module/notebook level are global in that module. They remain bound until reassigned, deleted, or the runtime ends. Functions can read them when no nearer binding shadows them.

Rebinding a global name inside a function requires `global`; mutating a shared list or dictionary does not. Prefer passing data as parameters so the dependency is visible.


**Figure 4.1: Reading global variables inside functions**

Assume non-negative radius and time. The fall model starts from rest, uses constant downward acceleration 9.81 m/s², ignores air resistance, and applies before impact.

In [10]:
PI = 3.14159
GRAVITY = 9.81
def circle_area(radius):
    return PI * radius ** 2
def fall_distance(time):
    return 0.5 * GRAVITY * time ** 2
print(f"Circle area: {circle_area(5):.2f}")
print(f"Fall distance: {fall_distance(3):.2f}")


Circle area: 78.54
Fall distance: 44.15


**Line-by-line walkthrough**

- **Line 1 —** `` PI = 3.14159 ``

  Define a shared approximation of pi. Uppercase naming is a convention, not enforced immutability.

- **Line 2 —** `` GRAVITY = 9.81 ``

  Define gravitational acceleration as 9.81 m/s².

- **Line 3 —** `` def circle_area(radius): ``

  Define a circle-area function with radius as its explicit input.

- **Line 4 —** `` return PI * radius ** 2 ``

  Read the surrounding PI and return area in squared radius units; assume a nonnegative radius.

- **Line 5 —** `` def fall_distance(time): ``

  Define a falling-distance model with time in seconds.

- **Line 6 —** `` return 0.5 * GRAVITY * time ** 2 ``

  Return the distance fallen from rest under constant gravity, neglecting air resistance. The model assumes nonnegative elapsed time.

- **Line 7 —** `` print(f"Circle area: {circle_area(5):.2f}") ``

  Print the area for radius 5 to two decimals.

- **Line 8 —** `` print(f"Fall distance: {fall_distance(3):.2f}") ``

  Print the model distance at three seconds to two decimals.



**Figure 4.2: Trying to modify a global variable (this creates a NEW local variable!)**

In [11]:
score = 100
def add_points():
    score = 150
    print(f"Inside function: score = {score}")
add_points()
print(f"Outside function: score = {score}")


Inside function: score = 150
Outside function: score = 100


**Line-by-line walkthrough**

- **Line 1 —** `` score = 100 ``

  Set score to 100 in the surrounding namespace.

- **Line 2 —** `` def add_points(): ``

  Define a function with no parameters.

- **Line 3 —** `` score = 150 ``

  Assign a separate local score of 150; no global declaration connects this assignment to the surrounding binding.

- **Line 4 —** `` print(f"Inside function: score = {score}") ``

  Print the local score.

- **Line 5 —** `` add_points() ``

  Call the function and display 150 from inside it.

- **Line 6 —** `` print(f"Outside function: score = {score}") ``

  Print the surrounding score, still 100.



> 💡 **Note:** Assigning to a name in a function normally makes that name local throughout the function. `global` and `nonlocal` declarations are exceptions. Changing `items[0]` or calling `items.append(...)` changes an object rather than rebinding the name `items`.


---
### ⏱️ Checkpoint 2 of 5 — Globals (target 01:55)

For reusable functions, should global state usually be avoided? Answer yes or no.

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [12]:
checkpoint_2_answer = ""
check_answer(
    2, checkpoint_2_answer, 'yes',
    'Explicit inputs and outputs are easier to test.',
)


Checkpoint 2: enter your prediction, then run again.


**Line-by-line walkthrough**

- **Line 1 —** `` checkpoint_2_answer = "" ``

  Store your response as text; replace the empty string before running the check.

- **Line 2 —** `` check_answer( ``

  Begin a call to the study helper defined earlier.

- **Line 3 —** `` 2, checkpoint_2_answer, 'yes', ``

  Supply checkpoint 2, your response and expected answer text `yes`.

- **Line 4 —** `` 'Explicit inputs and outputs are easier to test.', ``

  Supply the feedback explanation: Explicit inputs and returned results are easier to test.

- **Line 5 —** `` ) ``

  Close the call to print feedback and record the result.



---
## Part 5: The `global` Keyword

The `global` keyword tells Python: "I want to use the **global** variable, not create a new local one."

**Figure 5.1: Using the `global` keyword**

In [13]:
total_score = 0
def add_score(points):
    global total_score
    total_score += points
add_score(10)
add_score(25)
add_score(15)
print(f"Total: {total_score}")


Total: 50


**Line-by-line walkthrough**

- **Line 1 —** `` total_score = 0 ``

  Initialize a score in the surrounding namespace.

- **Line 2 —** `` def add_score(points): ``

  Define a function that changes that shared score.

- **Line 3 —** `` global total_score ``

  Declare that assignments to total_score in this function refer to the surrounding global binding.

- **Line 4 —** `` total_score += points ``

  Increase the shared score by this call's points.

- **Line 5 —** `` add_score(10) ``

  Add ten, making the shared total ten.

- **Line 6 —** `` add_score(25) ``

  Add twenty-five, bringing it to thirty-five.

- **Line 7 —** `` add_score(15) ``

  Add fifteen, bringing it to fifty.

- **Line 8 —** `` print(f"Total: {total_score}") ``

  Print the accumulated shared total; its value depends on preceding calls.



**Figure 5.2: Why `global` is usually a bad idea**

In [14]:
counter = 0
def increment_bad():
    global counter
    counter += 1
def increment_good(counter):
    return counter + 1
my_counter = 0
my_counter = increment_good(my_counter)
my_counter = increment_good(my_counter)
print(f"Counter: {my_counter}")


Counter: 2


**Line-by-line walkthrough**

- **Line 1 —** `` counter = 0 ``

  Initialize the shared counter used by the first design.

- **Line 2 —** `` def increment_bad(): ``

  Define a function that relies on shared state.

- **Line 3 —** `` global counter ``

  Declare the counter global so assignment targets the surrounding binding.

- **Line 4 —** `` counter += 1 ``

  Increase that global counter when called; this definition alone does not execute the increment.

- **Line 5 —** `` def increment_good(counter): ``

  Define an alternative with the current counter passed explicitly.

- **Line 6 —** `` return counter + 1 ``

  Return the next value without mutating any surrounding binding.

- **Line 7 —** `` my_counter = 0 ``

  Initialize the caller's independent counter.

- **Line 8 —** `` my_counter = increment_good(my_counter) ``

  Call the returning function and assign its result back to my_counter, making it one.

- **Line 9 —** `` my_counter = increment_good(my_counter) ``

  Repeat to make it two.

- **Line 10 —** `` print(f"Counter: {my_counter}") ``

  Print two. The explicit input and result make this version easier to test in isolation.



> 💡 **Note:** Avoid `global` in most cases. Instead, pass values as **parameters** and use **return** to send results back. This makes your code easier to understand, test, and debug. Names such as `PI` and `GRAVITY` are constants by convention: Python does not prevent reassignment. Shared constants can be useful when the dependency is clear and the program leaves them unchanged.

---
## Part 6: Variable Shadowing

**Variable shadowing** occurs when a local variable has the **same name** as a global variable. The local variable "shadows" (hides) the global one inside the function.

**Figure 6.1: Shadowing in action**

In [15]:
message = "Hello from global!"
def show_message():
    message = "Hello from local!"
    print(message)
show_message()
print(message)


Hello from local!
Hello from global!


**Line-by-line walkthrough**

- **Line 1 —** `` message = "Hello from global!" ``

  Store a message in the surrounding namespace.

- **Line 2 —** `` def show_message(): ``

  Define a function with its own scope.

- **Line 3 —** `` message = "Hello from local!" ``

  Create a local message that shadows the surrounding name.

- **Line 4 —** `` print(message) ``

  Print the local text because this binding takes precedence inside the function.

- **Line 5 —** `` show_message() ``

  Call the function to print the local message.

- **Line 6 —** `` print(message) ``

  Print the surrounding message, which the local assignment did not change.



**Figure 6.2: A tricky scope error — UnboundLocalError**

In [16]:
value = 10
def tricky():
    try:
        print(value)
        value = 20
    except UnboundLocalError:
        print("Error: tried to read local variable before assigning it!")
tricky()


Error: tried to read local variable before assigning it!


**Line-by-line walkthrough**

- **Line 1 —** `` value = 10 ``

  Set value to ten in the surrounding namespace.

- **Line 2 —** `` def tricky(): ``

  Define a function demonstrating a local-before-assignment error.

- **Line 3 —** `` try: ``

  Begin a try block so the expected error can be handled within the example.

- **Line 4 —** `` print(value) ``

  Attempt to read local value before it has been assigned, raising UnboundLocalError rather than using the surrounding value.

- **Line 5 —** `` value = 20 ``

  This assignment makes value local throughout the function, even though execution never reaches this line after the preceding error.

- **Line 6 —** `` except UnboundLocalError: ``

  Catch the specific UnboundLocalError.

- **Line 7 —** `` print("Error: tried to read local variable before assigning it!") ``

  Print an explanation and allow the function to end normally.

- **Line 8 —** `` tricky() ``

  Call the demonstration; the error is caught, so the cell finishes instead of stopping with an uncaught exception.



> 💡 **Note:** An assignment to a name in the function body normally makes it local throughout that function, unless declared `global` or `nonlocal`. Reading it before it has a value raises `UnboundLocalError`; Python does not fall back to the global. The `try/except` above catches this deliberate error so Run All continues; error handling is covered next week.


---
## Part 7: Functions Calling Functions

Functions can call other functions! This is called **composition** — building complex behavior from simple building blocks.

**Figure 7.1: Simple function composition**

In [17]:
def square(x):
    return x ** 2
def add(a, b):
    return a + b
def sum_of_squares(a, b):
    """Calculate a² + b² using the functions above."""
    return add(square(a), square(b))
print(sum_of_squares(3, 4))


25


**Line-by-line walkthrough**

- **Line 1 —** `` def square(x): ``

  Define a helper that accepts one value.

- **Line 2 —** `` return x ** 2 ``

  Return its square without printing.

- **Line 3 —** `` def add(a, b): ``

  Define an addition helper with two parameters.

- **Line 4 —** `` return a + b ``

  Return their sum.

- **Line 5 —** `` def sum_of_squares(a, b): ``

  Define a function combining the two helpers.

- **Line 6 —** `` """Calculate a² + b² using the functions above.""" ``

  Document the intended sum of squares.

- **Line 7 —** `` return add(square(a), square(b)) ``

  Calculate each square, pass those returned values into add, and return the resulting sum.

- **Line 8 —** `` print(sum_of_squares(3, 4)) ``

  Call the composed function with 3 and 4, then print 25.



---
### ⏱️ Checkpoint 3 of 5 — Composition (target 02:55)

What do we call one function using another function?

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [18]:
checkpoint_3_answer = ""
check_answer(
    3, checkpoint_3_answer, 'composition',
    'Function composition builds larger behavior from small units.',
)


Checkpoint 3: enter your prediction, then run again.


**Line-by-line walkthrough**

- **Line 1 —** `` checkpoint_3_answer = "" ``

  Store your response as text; replace the empty string before running the check.

- **Line 2 —** `` check_answer( ``

  Begin a call to the study helper defined earlier.

- **Line 3 —** `` 3, checkpoint_3_answer, 'composition', ``

  Supply checkpoint 3, your response and expected answer text `composition`.

- **Line 4 —** `` 'Function composition builds larger behavior from small units.', ``

  Supply the feedback explanation: Composition builds larger behavior by combining smaller functions.

- **Line 5 —** `` ) ``

  Close the call to print feedback and record the result.



**Figure 7.2: Building a student report with composition**

Use exam scores from 0 to 100. The score uses midterm 50% and final 50%. The letter thresholds are an illustrative programming rule, not a statement of university grading policy.


In [19]:
def calculate_average(midterm, final):
    """Return the course score: 50% midterm and 50% final."""
    return 0.5 * midterm + 0.5 * final
def determine_letter(average):
    """Convert numeric average to letter grade."""
    if average >= 90: return "AA"
    elif average >= 80: return "BA"
    elif average >= 70: return "BB"
    elif average >= 60: return "CB"
    elif average >= 50: return "CC"
    else: return "FF"
def pass_or_fail(letter):
    """Determine if the student passed."""
    return "PASS" if letter != "FF" else "FAIL"
def generate_report(name, midterm, final):
    """Generate a full report — uses all three functions above."""
    avg = calculate_average(midterm, final)
    letter = determine_letter(avg)
    status = pass_or_fail(letter)
    return f"{name}: {avg:.1f} ({letter}) - {status}"
print(generate_report("Elif Kaya", 85, 92))
print(generate_report("Burak Demir", 40, 35))


Elif Kaya: 88.5 (BA) - PASS
Burak Demir: 37.5 (FF) - FAIL


**Line-by-line walkthrough**

- **Line 1 —** `` def calculate_average(midterm, final): ``

  Define the course-score calculation from midterm and final.

- **Line 2 —** `` """Return the course score: 50% midterm and 50% final.""" ``

  Document the equal-weight assessment rule.

- **Line 3 —** `` return 0.5 * midterm + 0.5 * final ``

  Return 50% midterm plus 50% final.

- **Line 4 —** `` def determine_letter(average): ``

  Define a conversion from numerical score to the example's letter categories.

- **Line 5 —** `` """Convert numeric average to letter grade.""" ``

  Document that conversion.

- **Line 6 —** `` if average >= 90: return "AA" ``

  Return AA for scores of at least 90.

- **Line 7 —** `` elif average >= 80: return "BA" ``

  Otherwise return BA for scores of at least 80.

- **Line 8 —** `` elif average >= 70: return "BB" ``

  Otherwise return BB for scores of at least 70.

- **Line 9 —** `` elif average >= 60: return "CB" ``

  Otherwise return CB for scores of at least 60.

- **Line 10 —** `` elif average >= 50: return "CC" ``

  Otherwise return CC for scores of at least 50.

- **Line 11 —** `` else: return "FF" ``

  Return FF when no passing boundary matched.

- **Line 12 —** `` def pass_or_fail(letter): ``

  Define a status conversion from the letter category.

- **Line 13 —** `` """Determine if the student passed.""" ``

  Document the function's pass/fail role.

- **Line 14 —** `` return "PASS" if letter != "FF" else "FAIL" ``

  Return FAIL for FF and PASS for all other supplied letters.

- **Line 15 —** `` def generate_report(name, midterm, final): ``

  Define a report builder with name and both exam scores as explicit inputs.

- **Line 16 —** `` """Generate a full report — uses all three functions above.""" ``

  Document that it combines the three preceding helpers.

- **Line 17 —** `` avg = calculate_average(midterm, final) ``

  Calculate the equal-weight numerical score.

- **Line 18 —** `` letter = determine_letter(avg) ``

  Convert that returned score to a letter.

- **Line 19 —** `` status = pass_or_fail(letter) ``

  Convert the letter to a status.

- **Line 20 —** `` return f"{name}: {avg:.1f} ({letter}) - {status}" ``

  Return one formatted report string, displaying the score to one decimal place.

- **Line 21 —** `` print(generate_report("Elif Kaya", 85, 92)) ``

  Build and print Elif's report: 88.5, BA, PASS.

- **Line 22 —** `` print(generate_report("Burak Demir", 40, 35)) ``

  Build and print Burak's report: 37.5, FF, FAIL.



> 💡 **Note:** Each function does **one thing well**. Then we combine them to build more complex behavior. This makes code easier to test, debug, and reuse.

---
## Part 8: Organizing Code with Functions — The Mini-Library Approach

A **mini-library** is a collection of related functions that work together. Think of it as a toolbox for a specific task.

Good library design principles:

| Principle | Description |
|-----------|-------------|
| **Single Responsibility** | Each function does one thing |
| **Clear Naming** | Function name describes what it does |
| **Docstrings** | Each function has a description |
| **Consistent Interface** | Similar functions work in similar ways |
| **Explicit Effects** | Prefer returned results; document intentional printing, file writing, or mutation of inputs |

**Figure 8.1: A mini-library for text analysis**

Here a “word” means a whitespace-separated token; punctuation contributes to its length. Sentence count is only a punctuation heuristic: `...`, abbreviations and `?!` can mislead it. Empty text returns zero counts; zero averages are display conventions for no data.


In [20]:
def count_words(text):
    """Count the number of words in a text."""
    return len(text.split())
def count_sentences(text):
    """Estimate sentences by counting . ! ? marks; not linguistic parsing."""
    if not text.strip():
        return 0
    count = 0
    for char in text:
        if char in ".!?":
            count += 1
    return max(count, 1)
def average_word_length(text):
    """Calculate the average length of words."""
    words = text.split()
    total_length = sum(len(word) for word in words)
    return total_length / len(words) if words else 0
def text_summary(text):
    """Generate a summary using all helper functions."""
    words = count_words(text)
    sentences = count_sentences(text)
    avg_len = average_word_length(text)
    ratio = words / sentences if sentences else 0
    return (
        f"Words: {words}\n"
        f"Estimated sentences: {sentences}\n"
        f"Avg word length: {avg_len:.1f}\n"
        f"Words per sentence: {ratio:.1f}"
    )
sample = "Python is great. It is easy to learn! Do you agree?"
print(text_summary(sample))


Words: 11
Estimated sentences: 3
Avg word length: 3.7
Words per sentence: 3.7


**Line-by-line walkthrough**

- **Line 1 —** `` def count_words(text): ``

  Define a word-count helper accepting text.

- **Line 2 —** `` """Count the number of words in a text.""" ``

  Document its counting purpose.

- **Line 3 —** `` return len(text.split()) ``

  Split on whitespace and return the number of nonempty tokens.

- **Line 4 —** `` def count_sentences(text): ``

  Define a simple sentence-count estimate.

- **Line 5 —** `` """Estimate sentences by counting . ! ? marks; not linguistic parsing.""" ``

  Document that punctuation counting is a heuristic, not linguistic parsing.

- **Line 6 —** `` if not text.strip(): ``

  Check for empty or whitespace-only text.

- **Line 7 —** `` return 0 ``

  Return zero for that no-content case.

- **Line 8 —** `` count = 0 ``

  Initialize the punctuation count.

- **Line 9 —** `` for char in text: ``

  Visit each character.

- **Line 10 —** `` if char in ".!?": ``

  Treat each period, exclamation mark or question mark as a boundary marker.

- **Line 11 —** `` count += 1 ``

  Increase the estimate for each marker, even repeated punctuation.

- **Line 12 —** `` return max(count, 1) ``

  Return at least one for nonblank text, including text with no terminal punctuation.

- **Line 13 —** `` def average_word_length(text): ``

  Define a mean-word-length helper.

- **Line 14 —** `` """Calculate the average length of words.""" ``

  Document its purpose.

- **Line 15 —** `` words = text.split() ``

  Split on whitespace; punctuation remains part of each token.

- **Line 16 —** `` total_length = sum(len(word) for word in words) ``

  Sum token lengths using a generator expression.

- **Line 17 —** `` return total_length / len(words) if words else 0 ``

  Return total length divided by token count, or the chosen zero convention for empty text.

- **Line 18 —** `` def text_summary(text): ``

  Define a summary that composes the helpers.

- **Line 19 —** `` """Generate a summary using all helper functions.""" ``

  Document its report-building role.

- **Line 20 —** `` words = count_words(text) ``

  Calculate the word count.

- **Line 21 —** `` sentences = count_sentences(text) ``

  Calculate the sentence estimate.

- **Line 22 —** `` avg_len = average_word_length(text) ``

  Calculate the mean token length.

- **Line 23 —** `` ratio = words / sentences if sentences else 0 ``

  Calculate words per estimated sentence only when the denominator is nonzero; otherwise use zero.

- **Line 24 —** `` return ( ``

  Begin a parenthesized return expression spanning several lines.

- **Line 25 —** `` f"Words: {words}\n" ``

  Add the word-count line and a newline to the returned string.

- **Line 26 —** `` f"Estimated sentences: {sentences}\n" ``

  Add the sentence-estimate line and a newline.

- **Line 27 —** `` f"Avg word length: {avg_len:.1f}\n" ``

  Add the formatted mean-word-length line and a newline.

- **Line 28 —** `` f"Words per sentence: {ratio:.1f}" ``

  Add the final formatted words-per-sentence line. Adjacent f-strings are combined into one string.

- **Line 29 —** `` ) ``

  Close the return expression.

- **Line 30 —** `` sample = "Python is great. It is easy to learn! Do you agree?" ``

  Provide sample text with three boundary marks.

- **Line 31 —** `` print(text_summary(sample)) ``

  Build and print the complete multiline summary.



---
## Part 9: Refactoring — From Script to Functions

**Refactoring** means restructuring code without changing what it does. Let's take a flat script and break it into functions.

**Figure 9.1: BEFORE refactoring — everything in one block**

Use exam scores from 0 to 100. The score uses midterm 50% and final 50%. The letter thresholds are an illustrative programming rule, not a statement of university grading policy.


In [21]:
students = [
    {"name": "Zeynep", "midterm": 85, "final": 90},
    {"name": "Omer", "midterm": 45, "final": 50},
    {"name": "Selin", "midterm": 92, "final": 88},
]
for s in students:
    avg = 0.5 * s["midterm"] + 0.5 * s["final"]
    if avg >= 90: letter = "AA"
    elif avg >= 80: letter = "BA"
    elif avg >= 70: letter = "BB"
    elif avg >= 60: letter = "CB"
    elif avg >= 50: letter = "CC"
    else: letter = "FF"
    if letter != "FF": status = "PASS"
    else: status = "FAIL"
    print(f"{s['name']}: {avg:.1f} ({letter}) - {status}")


Zeynep: 87.5 (BA) - PASS
Omer: 47.5 (FF) - FAIL
Selin: 90.0 (AA) - PASS


**Line-by-line walkthrough**

- **Line 1 —** `` students = [ ``

  Begin the example list of student records.

- **Line 2 —** `` {"name": "Zeynep", "midterm": 85, "final": 90}, ``

  Store Zeynep's name and two exam scores.

- **Line 3 —** `` {"name": "Omer", "midterm": 45, "final": 50}, ``

  Store Omer's record.

- **Line 4 —** `` {"name": "Selin", "midterm": 92, "final": 88}, ``

  Store Selin's record.

- **Line 5 —** `` ] ``

  Close the list.

- **Line 6 —** `` for s in students: ``

  Visit one record per pass.

- **Line 7 —** `` avg = 0.5 * s["midterm"] + 0.5 * s["final"] ``

  Calculate its equal-weight exam average inline.

- **Line 8 —** `` if avg >= 90: letter = "AA" ``

  Assign AA when the score is at least 90.

- **Line 9 —** `` elif avg >= 80: letter = "BA" ``

  Otherwise assign BA when it is at least 80.

- **Line 10 —** `` elif avg >= 70: letter = "BB" ``

  Otherwise assign BB when it is at least 70.

- **Line 11 —** `` elif avg >= 60: letter = "CB" ``

  Otherwise assign CB when it is at least 60.

- **Line 12 —** `` elif avg >= 50: letter = "CC" ``

  Otherwise assign CC when it is at least 50.

- **Line 13 —** `` else: letter = "FF" ``

  Assign FF for lower scores.

- **Line 14 —** `` if letter != "FF": status = "PASS" ``

  Assign PASS for a non-FF result.

- **Line 15 —** `` else: status = "FAIL" ``

  Otherwise assign FAIL.

- **Line 16 —** `` print(f"{s['name']}: {avg:.1f} ({letter}) - {status}") ``

  Print the complete report. The next example reorganizes this behavior into functions without changing its output.



**Figure 9.2: AFTER refactoring — clean, reusable functions**

In [22]:
def calc_average(midterm, final):
    """Return 50% midterm plus 50% final."""
    return 0.5 * midterm + 0.5 * final
def get_letter_grade(average):
    """Convert numeric average to letter grade."""
    if average >= 90: return "AA"
    elif average >= 80: return "BA"
    elif average >= 70: return "BB"
    elif average >= 60: return "CB"
    elif average >= 50: return "CC"
    else: return "FF"
def get_status(letter):
    """Determine pass/fail from letter grade."""
    return "PASS" if letter != "FF" else "FAIL"
def print_report(students):
    """Print grade report for all students."""
    for s in students:
        avg = calc_average(s["midterm"], s["final"])
        letter = get_letter_grade(avg)
        status = get_status(letter)
        print(f"{s['name']}: {avg:.1f} ({letter}) - {status}")
students = [
    {"name": "Zeynep", "midterm": 85, "final": 90},
    {"name": "Omer", "midterm": 45, "final": 50},
    {"name": "Selin", "midterm": 92, "final": 88},
]
print_report(students)


Zeynep: 87.5 (BA) - PASS
Omer: 47.5 (FF) - FAIL
Selin: 90.0 (AA) - PASS


**Line-by-line walkthrough**

- **Line 1 —** `` def calc_average(midterm, final): ``

  Define the course-score calculation from midterm and final.

- **Line 2 —** `` """Return 50% midterm plus 50% final.""" ``

  Document the equal-weight assessment rule.

- **Line 3 —** `` return 0.5 * midterm + 0.5 * final ``

  Return 50% midterm plus 50% final.

- **Line 4 —** `` def get_letter_grade(average): ``

  Define a conversion from numerical score to the example's letter categories.

- **Line 5 —** `` """Convert numeric average to letter grade.""" ``

  Document that conversion.

- **Line 6 —** `` if average >= 90: return "AA" ``

  Return AA for scores of at least 90.

- **Line 7 —** `` elif average >= 80: return "BA" ``

  Otherwise return BA for scores of at least 80.

- **Line 8 —** `` elif average >= 70: return "BB" ``

  Otherwise return BB for scores of at least 70.

- **Line 9 —** `` elif average >= 60: return "CB" ``

  Otherwise return CB for scores of at least 60.

- **Line 10 —** `` elif average >= 50: return "CC" ``

  Otherwise return CC for scores of at least 50.

- **Line 11 —** `` else: return "FF" ``

  Return FF when no passing boundary matched.

- **Line 12 —** `` def get_status(letter): ``

  Define a status conversion from the letter category.

- **Line 13 —** `` """Determine pass/fail from letter grade.""" ``

  Document the function's pass/fail role.

- **Line 14 —** `` return "PASS" if letter != "FF" else "FAIL" ``

  Return FAIL for FF and PASS for all other supplied letters.

- **Line 15 —** `` def print_report(students): ``

  Define a display function for a list of student records.

- **Line 16 —** `` """Print grade report for all students.""" ``

  Document that this function prints reports rather than returning report strings.

- **Line 17 —** `` for s in students: ``

  Visit each supplied student record.

- **Line 18 —** `` avg = calc_average(s["midterm"], s["final"]) ``

  Pass its exam scores into the equal-weight helper.

- **Line 19 —** `` letter = get_letter_grade(avg) ``

  Convert the returned average to a letter with the dedicated helper.

- **Line 20 —** `` status = get_status(letter) ``

  Convert the letter to a status.

- **Line 21 —** `` print(f"{s['name']}: {avg:.1f} ({letter}) - {status}") ``

  Print the same name, average, letter and status as the earlier inline version.

- **Line 22 —** `` students = [ ``

  Begin the same demonstration dataset used before refactoring.

- **Line 23 —** `` {"name": "Zeynep", "midterm": 85, "final": 90}, ``

  Store Zeynep's unchanged record.

- **Line 24 —** `` {"name": "Omer", "midterm": 45, "final": 50}, ``

  Store Omer's unchanged record.

- **Line 25 —** `` {"name": "Selin", "midterm": 92, "final": 88}, ``

  Store Selin's unchanged record.

- **Line 26 —** `` ] ``

  Close the list.

- **Line 27 —** `` print_report(students) ``

  Call the report function once to process all three students.



> 💡 **Note:** After refactoring, each function can be tested individually. Need to change how letter grades work? Just edit `get_letter_grade()` — nothing else needs to change.

---
## Part 10: Putting It Together — A Sensor Mini-Library

Return to the two sensors from the opening problem. Pass calibration settings explicitly, convert each voltage, then format a report. Processing B must not change A’s result.


**Figure 10.1: One library, two sensor calibrations**

Assume finite numeric values, positive gain and a linear response over 0–100 N. The valid voltage interval is `offset` through `offset + 100 / gain`. Outside it, return `None`. Sensor A supports 0.5–4.5 V; B supports 1.0–5.0 V. The settings dictionaries are read without mutation.


In [23]:
def sensor_force(voltage, offset, gain):
    """Return calibrated force (N), or None."""
    if not offset <= voltage <= offset + 100 / gain:
        return None
    return (voltage - offset) * gain
def sensor_report(name, voltage, settings):
    """Return text without changing settings."""
    force = sensor_force(voltage, settings["offset"], settings["gain"])
    if force is None:
        return f"{name}: outside calibrated range"
    return f"{name}: {force:.1f} N"
a = {"offset": 0.5, "gain": 25}
b = {"offset": 1.0, "gain": 25}
print(sensor_report("A", 2.5, a))
print(sensor_report("B", 2.5, b))
print(sensor_report("A", 2.5, a))


A: 50.0 N
B: 37.5 N
A: 50.0 N


**Line-by-line walkthrough**

- **Line 1 —** `` def sensor_force(voltage, offset, gain): ``

  Define force conversion using voltage, offset and gain as explicit inputs.

- **Line 2 —** `` """Return calibrated force (N), or None.""" ``

  Document the force result in newtons or the None marker.

- **Line 3 —** `` if not offset <= voltage <= offset + 100 / gain: ``

  Check the calibrated 0–100 N interval expressed in volts. This model assumes a finite positive gain and valid numerical inputs; zero gain would cause division by zero.

- **Line 4 —** `` return None ``

  Return None for an out-of-range voltage.

- **Line 5 —** `` return (voltage - offset) * gain ``

  Subtract the offset and multiply by the gain to return force in newtons.

- **Line 6 —** `` def sensor_report(name, voltage, settings): ``

  Define a report builder receiving a sensor name, voltage and settings dictionary.

- **Line 7 —** `` """Return text without changing settings.""" ``

  Document that the settings are read without mutation.

- **Line 8 —** `` force = sensor_force(voltage, settings["offset"], settings["gain"]) ``

  Pass the voltage and this sensor's offset and gain to the calculation helper.

- **Line 9 —** `` if force is None: ``

  Check specifically for an invalid None result, preserving a valid zero force.

- **Line 10 —** `` return f"{name}: outside calibrated range" ``

  Return an out-of-range report immediately for rejected input.

- **Line 11 —** `` return f"{name}: {force:.1f} N" ``

  Otherwise return the labelled force to one decimal place.

- **Line 12 —** `` a = {"offset": 0.5, "gain": 25} ``

  Create sensor A's calibration settings.

- **Line 13 —** `` b = {"offset": 1.0, "gain": 25} ``

  Create sensor B's different offset with the same gain.

- **Line 14 —** `` print(sensor_report("A", 2.5, a)) ``

  Build and print A's report at 2.5 V, giving 50.0 N.

- **Line 15 —** `` print(sensor_report("B", 2.5, b)) ``

  Build and print B's report at the same voltage, giving 37.5 N.

- **Line 16 —** `` print(sensor_report("A", 2.5, a)) ``

  Call A again to show that using B's settings did not alter A's result or settings.



---
### ⏱️ Checkpoint 4 of 5 — Shadowing (target 03:55)

Can a local variable have the same name as a global variable? Answer yes or no.

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [24]:
checkpoint_4_answer = ""
check_answer(
    4, checkpoint_4_answer, 'yes',
    'The local name shadows the global name inside the function.',
)


Checkpoint 4: enter your prediction, then run again.


**Line-by-line walkthrough**

- **Line 1 —** `` checkpoint_4_answer = "" ``

  Store your response as text; replace the empty string before running the check.

- **Line 2 —** `` check_answer( ``

  Begin a call to the study helper defined earlier.

- **Line 3 —** `` 4, checkpoint_4_answer, 'yes', ``

  Supply checkpoint 4, your response and expected answer text `yes`.

- **Line 4 —** `` 'The local name shadows the global name inside the function.', ``

  Supply the feedback explanation: A local binding shadows the surrounding name inside the function.

- **Line 5 —** `` ) ``

  Close the call to print feedback and record the result.



---
## Exercises — Problems to Solve

> Each exercise is a problem. Think about the **STEPS** before you code. Decompose the problem, plan your approach, then implement it.

Complete the exercises below. Each exercise cell starts with `# ✏️ [EXn]` — **do not remove this line**.

### Core Practice and Optional Extension

- **Exercises 1–8:** core in-class practice.
- **Exercises 9 and above:** optional extension; these are not homework.
- At Checkpoint 5, list the exercises you have tested and can explain. This is a self-report, not automatic grading.


### Start with scope, then build small libraries

For each exercise, state the inputs, the returned value and any printing or mutation. Test a changed input before comparing with the worked solution.


### Exercise 1: Scope Prediction (Easy)

Without running the code first, **predict** what the following program will print. Then add `print()` statements in the cell below to show the output.

```python
x = 10

def foo():
    x = 20
    print(x)

def bar():
    print(x)

foo()
bar()
print(x)
```

**Expected output:**
```
20
10
10
```

<details><summary>💡 Hint</summary>

`foo()` creates a local `x = 20`, which shadows the global `x = 10`. `bar()` reads the global `x` because it has no local `x`. After both calls, the global `x` is still `10`.

</details>

In [ ]:
# ✏️ [EX1]
# Type the code from above and verify your prediction.
# Add a comment explaining WHY each print gives its value.


**Line-by-line walkthrough**

This is a practice starter. All displayed lines are comments beginning with `#`, so running this cell performs no calculation. Read the function requirements above, choose **Edit code**, and write your definition here. Add calls using the suggested examples, inspect the returned values, and compare with the worked solution after testing your own implementation. A function definition alone does not execute its body.


### Exercise 2: Fix the Scope Bug (Easy)

The following code has a **scope bug**. Find and fix it so that `get_full_greeting()` works correctly.

```python
def set_greeting():
    greeting = "Merhaba"

def get_full_greeting(name):
    return f"{greeting}, {name}!"

set_greeting()
print(get_full_greeting("Deniz"))
```

**Expected output:**
```
Merhaba, Deniz!
```

<details><summary>💡 Hint</summary>

The variable `greeting` is local to `set_greeting()`. Fix it by making `set_greeting()` return the greeting, or pass the greeting as a parameter to `get_full_greeting()`.

</details>

In [ ]:
# ✏️ [EX2]
# Fix the scope bug. Do NOT use the 'global' keyword.


**Line-by-line walkthrough**

This is a practice starter. All displayed lines are comments beginning with `#`, so running this cell performs no calculation. Read the function requirements above, choose **Edit code**, and write your definition here. Add calls using the suggested examples, inspect the returned values, and compare with the worked solution after testing your own implementation. A function definition alone does not execute its body.


### Exercise 3: Temperature Library (Medium)

Create a temperature conversion mini-library with these functions:

| Function | Formula |
|----------|---------|
| `c_to_f(celsius)` | F = C × 9/5 + 32 |
| `f_to_c(fahrenheit)` | C = (F − 32) × 5/9 |
| `c_to_k(celsius)` | K = C + 273.15 |
| `k_to_c(kelvin)` | C = K − 273.15 |

**Expected output:**
```
100°C = 212.0°F
32°F = 0.0°C
0°C = 273.15K
300K = 26.85°C
```

<details><summary>💡 Hint</summary>

Each function takes one number and returns one number. Use the formulas directly. Test each function with known values (e.g., 100°C = 212°F, 0°C = 273.15K).

</details>

**Input and result contract:** Assume finite temperature values at or above absolute zero (0 K, −273.15 °C, −459.67 °F). These helpers convert units; they do not validate physical plausibility.


In [ ]:
# ✏️ [EX3]
# Create your temperature conversion library below.


**Line-by-line walkthrough**

This is a practice starter. All displayed lines are comments beginning with `#`, so running this cell performs no calculation. Read the function requirements above, choose **Edit code**, and write your definition here. Add calls using the suggested examples, inspect the returned values, and compare with the worked solution after testing your own implementation. A function definition alone does not execute its body.


### Exercise 4: Math Helper Library (Medium)

Create a math helper library with these functions:

- `is_prime(n)` — returns `True` if `n` is prime, `False` otherwise
- `factorial(n)` — returns n! (e.g., factorial(5) = 120)
- `gcd(a, b)` — returns the greatest common divisor
- `lcm(a, b)` — returns the least common multiple (use `gcd` inside!)

**Expected output:**
```
is_prime(7) = True
is_prime(10) = False
factorial(5) = 120
gcd(12, 8) = 4
lcm(12, 8) = 24
```

<details><summary>💡 Hint</summary>

For `is_prime`: check divisibility from 2 to the square root of n. For `gcd`: use the Euclidean algorithm (while b != 0: a, b = b, a % b). For `lcm`: return 0 if either input is zero; otherwise use the formula `lcm(a,b) = abs(a*b) // gcd(a,b)`.

</details>

**Input and result contract:** Use integer inputs. Factorial requires n ≥ 0; the worked solution raises ValueError for a negative integer. GCD and LCM are non-negative even for negative arguments. Use gcd(0, 0) = 0 and return lcm = 0 if either argument is zero, before dividing by gcd.


In [ ]:
# ✏️ [EX4]
# Create your math helper library below.


**Line-by-line walkthrough**

This is a practice starter. All displayed lines are comments beginning with `#`, so running this cell performs no calculation. Read the function requirements above, choose **Edit code**, and write your definition here. Add calls using the suggested examples, inspect the returned values, and compare with the worked solution after testing your own implementation. A function definition alone does not execute its body.


### Exercise 5: String Helper Library (Medium)

Create a string helper library with these functions:

- `is_palindrome(text)` — returns `True` if text reads the same forwards and backwards (ignore case and spaces)
- `count_vowels(text)` — returns the count of vowels (a, e, i, o, u)
- `reverse_string(text)` — returns the reversed text

**Expected output:**
```
is_palindrome('racecar') = True
is_palindrome('hello') = False
count_vowels('Hello World') = 3
reverse_string('Python') = nohtyP
```

<details><summary>💡 Hint</summary>

For `is_palindrome`: convert to lowercase, remove spaces, then compare with reversed version. For `count_vowels`: loop through each character and check if it's in `"aeiouAEIOU"`. For `reverse_string`: use slicing `text[::-1]`.

</details>

**Input and result contract:** Ignore literal spaces and use Python lowercasing for the palindrome; punctuation and tabs remain significant. This is not Turkish locale-aware normalization. Count only the five English vowels, case-insensitively. Empty text is a palindrome by this exercise’s convention.


In [ ]:
# ✏️ [EX5]
# Create your string helper library below.


**Line-by-line walkthrough**

This is a practice starter. All displayed lines are comments beginning with `#`, so running this cell performs no calculation. Read the function requirements above, choose **Edit code**, and write your definition here. Add calls using the suggested examples, inspect the returned values, and compare with the worked solution after testing your own implementation. A function definition alone does not execute its body.


### Exercise 6: Statistics Library (Medium)

Create a statistics library with:

- `mean(numbers)` — returns the average
- `median(numbers)` — returns the middle value (sort first!)
- `mode(numbers)` — returns the most frequent value

**Expected output:**
```
Data: [4, 7, 2, 7, 3, 7, 5]
Mean: 5.0
Median: 5
Mode: 7
```

<details><summary>💡 Hint</summary>

For `median`: sort the list, then return the middle element (if odd length) or average of the two middle elements (if even length). For `mode`: count occurrences of each number using a dictionary or loop, then return the one with the highest count.

</details>

**Input contract:** use a nonempty list of numbers. For tied modes, return the value that appears first in the original list. Do not mutate the original list while finding the median. The bridge below deliberately explores an empty-list error.


In [ ]:
# ✏️ [EX6]
# Create your statistics library below.


**Line-by-line walkthrough**

This is a practice starter. All displayed lines are comments beginning with `#`, so running this cell performs no calculation. Read the function requirements above, choose **Edit code**, and write your definition here. Add calls using the suggested examples, inspect the returned values, and compare with the worked solution after testing your own implementation. A function definition alone does not execute its body.


### Exercise 7: Geometry Library (Medium)

Create a geometry library with:

- `circle_area(radius)` — returns π × r² (use `PI = 3.14159` as a global constant)
- `rect_area(width, height)` — returns width × height
- `triangle_area(base, height)` — returns 0.5 × base × height

Also create a `shape_report(shapes)` function that takes a list of shape dictionaries and prints a report.

**Expected output:**
```
circle (r=5): area = 78.54
rectangle (4x6): area = 24.00
triangle (b=3, h=8): area = 12.00
```

<details><summary>💡 Hint</summary>

Define `PI = 3.14159` as a global constant. Each shape dictionary could have a "type" key and the relevant dimensions. The `shape_report` function checks the type and calls the appropriate area function.

</details>

**Input and result contract:** Assume each shape has its required keys and non-negative dimensions in the same unit. Print the report and return the list of areas; an empty list prints nothing and returns []. An unknown type raises ValueError in the worked solution (error handling is next week).


In [ ]:
# ✏️ [EX7]
# Create your geometry library below.


**Line-by-line walkthrough**

This is a practice starter. All displayed lines are comments beginning with `#`, so running this cell performs no calculation. Read the function requirements above, choose **Edit code**, and write your definition here. Add calls using the suggested examples, inspect the returned values, and compare with the worked solution after testing your own implementation. A function definition alone does not execute its body.


### Exercise 8: Refactor — Inline to Functions (Medium)

Refactor the following inline code into well-organized functions. The output should remain **exactly the same**.

```python
# Inline code to refactor:
prices = [120, 250, 80, 310, 45, 190]
total = 0
discounted_total = 0
for p in prices:
    total += p
    if p > 200:
        discounted_total += p * 0.9
    elif p > 100:
        discounted_total += p * 0.95
    else:
        discounted_total += p
print(f"Original total: {total} TL")
print(f"Discounted total: {discounted_total:.2f} TL")
print(f"You saved: {total - discounted_total:.2f} TL")
```

**Expected output:**
```
Original total: 995 TL
Discounted total: 923.50 TL
You saved: 71.50 TL
```

<details><summary>💡 Hint</summary>

Create functions like `apply_discount(price)`, `calculate_total(prices)`, `calculate_discounted_total(prices)`, and `print_summary(prices)`. Each function should do one thing.

</details>

**Input and result contract:** Assume non-negative prices. Keep the strict > boundaries: 100 receives no discount, 200 receives 5%. An empty price list totals zero.


In [ ]:
# ✏️ [EX8]
# Refactor the inline code into functions below.


**Line-by-line walkthrough**

This is a practice starter. All displayed lines are comments beginning with `#`, so running this cell performs no calculation. Read the function requirements above, choose **Edit code**, and write your definition here. Add calls using the suggested examples, inspect the returned values, and compare with the worked solution after testing your own implementation. A function definition alone does not execute its body.


---
### Checkpoint 5 of 5 — Practice reflection (target 04:45)

After Exercises 1–8, edit `practiced_exercises` in the next cell.
List only the exercise numbers whose results you have tested and whose steps you can explain.
Leave the list empty until you have done that work. This is your explicit self-report;
the tool does not inspect or grade your solution and does not count execution history.

**Türkçe:** Bu liste öz değerlendirmedir. Hücreyi çalıştırmak tek başına yeterli değildir;
sonucu kontrol et ve çözüm adımlarını açıklayabildiğinden emin ol.


In [33]:
practiced_exercises = []
exercise_checkpoint(5, practiced_exercises, expected=8)
show_progress_summary()


Practice self-report: 0/8 core exercises reviewed.
This is your reflection, not a correctness score or a grade.
Still to review: 1, 2, 3, 4, 5, 6, 7, 8
For each exercise: test the result, explain the steps, then compare with the worked solution.

My study feedback (this runtime)
1. Concept check: 0/1
2. Concept check: 0/1
3. Concept check: 0/1
4. Concept check: 0/1
5. Practice self-report: 0/8
These checks send no grading submission. Save your notebook to keep your work.


**Line-by-line walkthrough**

- **Line 1 —** `` practiced_exercises = [] ``

  Start with an empty list of reviewed exercise numbers. Replace it with numbers such as `[1, 2]` only after you have tested and can explain those exercises; merely running the cell does not complete them.

- **Line 2 —** `` exercise_checkpoint(5, practiced_exercises, expected=8) ``

  Call the reflection helper for checkpoint 5 with your list and eight expected core exercises. It removes duplicates, checks the range and prints your reported progress; it does not examine solution correctness.

- **Line 3 —** `` show_progress_summary() ``

  Call the summary helper with no arguments to show the recorded results for checkpoints 1 through 5. Run the study-tools setup first so this function exists.



---
## 🌟 Optional Extension

Exercises 9 and above are optional enrichment. Stop here if the five-hour class has ended.


### Exercise 9: Validator Library (Medium)

Create a validator library with:

- `validate_age(age)` — returns `True` if age is between 0 and 150
- `validate_email(email)` — returns `True` if email contains `@` and `.` after `@`
- `validate_id(student_id)` — returns `True` if student_id is exactly 10 digits

Also create `validate_student(name, age, email, student_id)` that checks all three and returns a list of error messages (empty list if all valid).

**Expected output:**
```
Valid student: []
Invalid student: ['Invalid age', 'Invalid email']
```

<details><summary>💡 Hint</summary>

For `validate_email`: check if `"@"` is in the email, and if there's a `"."` after the `"@"`. For `validate_id`: check if `len(student_id) == 10` and all its characters are in `"0123456789"` (unlike `isdigit()`, this excludes Unicode digit symbols). The combined function should call each validator and collect error messages.

</details>

**Input and result contract:** Use numeric age from 0 through 150 inclusive; reject Boolean and nonnumeric values. ID is a string of exactly ten ASCII digits 0–9. The email check only requires @ and a dot after the first @; even "@." passes this classroom rule, so it is not a real email validator. The name is a label, not validated here.


In [ ]:
# ✏️ [EX9]
# Create your validator library below.


**Line-by-line walkthrough**

This is a practice starter. All displayed lines are comments beginning with `#`, so running this cell performs no calculation. Read the function requirements above, choose **Edit code**, and write your definition here. Add calls using the suggested examples, inspect the returned values, and compare with the worked solution after testing your own implementation. A function definition alone does not execute its body.


### Exercise 10: Composition (Medium)

Build a `process_grade(raw_score, bonus, curve)` function that:

1. Calls `apply_bonus(score, bonus)` — adds bonus points
2. Calls `apply_curve(score, curve)` — adds curve points
3. Calls `clamp(score, 0, 100)` — ensures score is between 0 and 100
4. Calls `to_letter(score)` — converts to letter grade

**Expected output:**
```
Raw: 72, Bonus: 5, Curve: 8 -> Score: 85, Grade: BA
Raw: 95, Bonus: 10, Curve: 5 -> Score: 100, Grade: AA
Raw: 30, Bonus: 0, Curve: 5 -> Score: 35, Grade: FF
```

<details><summary>💡 Hint</summary>

`clamp(value, min_val, max_val)` returns `min_val` if value < min_val, `max_val` if value > max_val, otherwise value. Use `min()` and `max()` built-in functions: `max(min_val, min(value, max_val))`.

</details>

**Exercise grading scale:** AA ≥90, BA ≥80, BB ≥70, CB ≥60, CC ≥50, otherwise FF; only FF fails. These are rules for this programming exercise, not the course assessment weights.


**Input and result contract:** Return (score, letter). Assume finite numeric values and ordered clamp limits. This bonus/curve pipeline is hypothetical function practice; CP1 has no bonus assessment component.


In [ ]:
# ✏️ [EX10]
# Build your composed grade processing function below.


**Line-by-line walkthrough**

This is a practice starter. All displayed lines are comments beginning with `#`, so running this cell performs no calculation. Read the function requirements above, choose **Edit code**, and write your definition here. Add calls using the suggested examples, inspect the returned values, and compare with the worked solution after testing your own implementation. A function definition alone does not execute its body.


### Exercise 11: Grade Report Generator (Challenge)

Build a complete grade report system that combines multiple helper functions. Given a list of students with their exam scores, generate a formatted report.

Requirements:
- Calculate weighted average (Midterm: 50%, Final: 50%)
- Determine letter grade and pass/fail status
- Calculate class statistics (highest, lowest, class average)
- Print a formatted report

**Test data:**
```python
students = [
    {"name": "Ayse Yildiz", "midterm": 78, "final": 85},
    {"name": "Kemal Ozturk", "midterm": 92, "final": 88},
    {"name": "Deniz Arslan", "midterm": 45, "final": 40},
    {"name": "Hakan Cetin", "midterm": 65, "final": 72},
    {"name": "Sibel Kara", "midterm": 88, "final": 95},
]
```

**Expected output:**
```
GRADE REPORT
Ayse Yildiz    : 81.5 (BA) - PASS
Kemal Ozturk   : 90.0 (AA) - PASS
Deniz Arslan   : 42.5 (FF) - FAIL
Hakan Cetin    : 68.5 (CB) - PASS
Sibel Kara     : 91.5 (AA) - PASS
Class Average: 74.8
Highest: 91.5 (Sibel Kara)
Lowest: 42.5 (Deniz Arslan)
Pass Rate: 80.0%
```

<details><summary>💡 Hint</summary>

Break it into functions: `weighted_average(midterm, final)`, `letter_grade(avg)`, `pass_fail(letter)`, `class_stats(student_results)`, `print_report(students)`. Build from the bottom up.

</details>

**Exercise grading scale:** AA ≥90, BA ≥80, BB ≥70, CB ≥60, CC ≥50, otherwise FF; only FF fails. These are rules for this programming exercise, not the course assessment weights.




**Input and result contract:** Assume complete student records with scores from 0 to 100. Empty results have no statistics and print "No student records." For tied highest/lowest scores, retain the first student.


In [ ]:
# ✏️ [EX11]
# Build your grade report generator below.


**Line-by-line walkthrough**

This is a practice starter. All displayed lines are comments beginning with `#`, so running this cell performs no calculation. Read the function requirements above, choose **Edit code**, and write your definition here. Add calls using the suggested examples, inspect the returned values, and compare with the worked solution after testing your own implementation. A function definition alone does not execute its body.


### Exercise 12: Physics Calculator (Challenge)

Create these small functions, then compose them in `physics_report(mass, distance, time, height)`.

| Function | Formula | Meaning |
|---|---|---|
| `velocity(distance, time)` | `distance / time` | **Average** speed for straight forward motion |
| `acceleration(v_final, v_initial, time)` | `(v_final - v_initial) / time` | Constant acceleration |
| `force(mass, acceleration)` | `mass * acceleration` | Net force along the motion |
| `kinetic_energy(mass, velocity)` | `0.5 * mass * velocity**2` | Energy at the stated instantaneous speed |
| `potential_energy(mass, height)` | `mass * 9.81 * height` | Gravitational energy relative to zero height |
| `momentum(mass, velocity)` | `mass * velocity` | Momentum along the motion |

**Physical model:** a 10 kg cart moves in a straight line from rest with constant
acceleration, covering 100 m in 5 s. Its height is 20 m above the chosen reference.
There is no reversal. Time is positive and mass is non-negative. The force requested
is the net force **along the motion**; height is only used to report gravitational energy.

The average speed is `100/5 = 20 m/s`. It is not the final speed.
From `distance = (v_initial + v_final)*time/2`, with `v_initial = 0`, obtain
`v_final = 2*distance/time = 40 m/s`, then `a = (40-0)/5 = 8 m/s²`.
Use the final speed for final kinetic energy and momentum.

**Expected output:**
```text
Average speed: 20.00 m/s
Final velocity: 40.00 m/s
Acceleration: 8.00 m/s²
Net force: 80.00 N
Final kinetic energy: 8000.00 J
Potential energy: 1962.00 J
Final momentum: 400.00 kg·m/s
```

**Türkçe:** `yol/zaman` ortalama sürattir. Durarak başlayıp sabit ivmeyle hızlanan
cisimde son sürat bunun iki katıdır. Kuvvet, enerji ve momentumda aynı fiziksel
modeli kullanın. Fonksiyonları kısa tutun; rapor fonksiyonu bunları sırayla çağırır.


**Input and result contract:** Use finite numeric inputs, distance ≥ 0 m, time > 0 s and mass ≥ 0 kg. Height in metres may be negative relative to the chosen reference. These helpers assume valid data; the bridge deliberately violates the time contract.


In [ ]:
# ✏️ [EX12]
# Create your physics calculator library below.


**Line-by-line walkthrough**

This is a practice starter. All displayed lines are comments beginning with `#`, so running this cell performs no calculation. Read the function requirements above, choose **Edit code**, and write your definition here. Add calls using the suggested examples, inspect the returned values, and compare with the worked solution after testing your own implementation. A function definition alone does not execute its body.


### Bridge Exercise: Preview of Error Handling

Next week, we'll learn about **error handling** with `try/except`. For now, try this: what happens when you call your physics calculator with `time = 0`? What about your statistics library with an empty list?

Write code that **demonstrates these errors** happening. We'll learn how to handle them gracefully next week!

**Expected behavior:**
```
velocity(100, 0) causes: ZeroDivisionError
mean([]) causes: ZeroDivisionError
```

<details><summary>💡 Hint</summary>

Define the two helpers first (from Exercises 6 and 12), then call them with invalid inputs. Without those definitions you get NameError instead. You can use `try/except` as a preview: `try: velocity(100, 0)` and `except ZeroDivisionError as e: print(e)`.

</details>

In [ ]:
# ✏️ [EXBridge]
# Experiment with calling functions with invalid inputs.
# What errors do you get? Can you predict them?


**Line-by-line walkthrough**

This is a practice starter. All displayed lines are comments beginning with `#`, so running this cell performs no calculation. Read the function requirements above, choose **Edit code**, and write your definition here. Add calls using the suggested examples, inspect the returned values, and compare with the worked solution after testing your own implementation. A function definition alone does not execute its body.


## Worked solutions and study support

Try each problem first. Then compare your reasoning and test cases with the complete
[Week 10 worked solutions](../solutions/Week_10_Solutions.ipynb).
The solution notebook includes every core exercise, optional exercise, and this week's bridge when present.

If you are using Colab, [open the published solution notebook](https://colab.research.google.com/github/ArifSolmaz/courses/blob/main/fall/cp1/solutions/Week_10_Solutions.ipynb).
Open it in a separate runtime. Running a solution first should not supply hidden variables to your own work.

**Türkçe:** Önce kendi çözümünü dene. Sonra adımları ve testleri karşılaştır; çözümü kapatıp farklı bir örneği kendin çöz.

[Simple course guide](../STUDY_GUIDE.md) · [All worked solutions](../solutions/README.md)
